In [1]:
import sys, json, time
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.common.config import PROCESSED, EXPERIMENTS, ask, ask_json
import src.eval.evaluate as ev
import pandas as pd
import re

VAL = ["P06", "P07"]                      # tune ONLY on validation
PROMPTS = ROOT / "src/methods/prompting/prompts"
PROMPTS.mkdir(parents=True, exist_ok=True)
EXPERIMENTS.mkdir(parents=True, exist_ok=True)
EVAL_TAG = "eval4" 

def inputs(prefix):
    p = ev.project_dir(prefix)
    return {"name": p.name,
            "brd": (p / "brd.md").read_text(encoding="utf-8"),
            "arch": (p / "architecture.md").read_text(encoding="utf-8")}

print({p: inputs(p)["name"] for p in VAL})

{'P06': 'P06_Rebate_Validation', 'P07': 'P07_Email_Order_Entry'}


In [2]:
def run_version(version, system, build_user=None, generate=None, projects=VAL, effort="medium", notes=""):
    """Either build_user(inp) -> user text (single call) or generate(inp) -> plan (custom pipeline)."""
    run_dir = EXPERIMENTS / f"prompting_{version}_{datetime.now():%Y%m%d_%H%M}"
    run_dir.mkdir(parents=True, exist_ok=True)
    example = build_user(inputs(projects[0]))[:3000] if build_user else "(custom generate function)"
    (run_dir / "prompt.json").write_text(json.dumps(
        {"version": version, "effort": effort, "notes": notes, "system": system, "user_example": example},
        indent=2), encoding="utf-8")

    def one(prefix):
        inp = inputs(prefix)
        t = time.time()
        plan = generate(inp) if generate else ask(system, build_user(inp), effort=effort)
        gen_s = time.time() - t
        (run_dir / f"{inp['name']}.md").write_text(plan, encoding="utf-8")
        r, cov = ev.score_plan(plan, prefix)
        cov.to_csv(run_dir / f"{inp['name']}_coverage.csv", index=False)
        (run_dir / f"{inp['name']}_scores.json").write_text(json.dumps(r, indent=2, default=str), encoding="utf-8")
        print(f"done {inp['name']}: gen {gen_s:.0f}s, {len(plan.split())} words", flush=True)
        return {"version": version, "run": run_dir.name, "project": inp["name"], "words": len(plan.split()),
                **ev.summary(r), "unverified_supported": r["unverified_supported"]}

    with ThreadPoolExecutor(max_workers=len(projects)) as pool:
        df = pd.DataFrame(list(pool.map(one, projects)))
    df["eval"] = EVAL_TAG
    path = EXPERIMENTS / "scores_by_plan.csv"
    df.to_csv(path, mode="a", header=not path.exists(), index=False)
    return df, run_dir

def leaderboard():
    cols = ["version", "overall", "structure", "coverage", "faithfulness", "invented", "similarity", "words", "notes"]
    b = pd.read_csv(EXPERIMENTS / "leaderboard.csv")
    return b[[c for c in cols if c in b.columns]]

def diagnose(run_dir, prefix):
    """What to fix next: missing sections, missed facts, invented claims, reviewer's missing points."""
    name = ev.project_dir(prefix).name
    r = json.loads((run_dir / f"{name}_scores.json").read_text(encoding="utf-8"))
    print("Missing sections:", r["missing_sections"])
    print("\nCoverage by category:", r["coverage_by_category"])
    print("\nInvented:"); [print(" -", c) for c in r["invented_claims"]]
    print("\nMissed must-test facts:"); [print(f" - [{f['category']}] {f['fact']}") for f in r["missed_facts"][:20]]
    print("\nReviewer – missing points by section:")
    for k, pts in list(r["missing_points"].items())[:10]:
        print(f" {k}: {pts[:3]}")

In [3]:
def rescore(run_dir, version, notes=""):
    """Re-score saved plans in run_dir with the current evaluator (no regeneration)."""
    def one(md_path):
        prefix = md_path.stem[:3]
        plan = md_path.read_text(encoding="utf-8")
        result, cov_df = ev.score_plan(plan, prefix)
        cov_df.to_csv(run_dir / f"{md_path.stem}_coverage.csv", index=False)
        (run_dir / f"{md_path.stem}_scores.json").write_text(json.dumps(result, indent=2, default=str), encoding="utf-8")
        return {**ev.summary(result), "project": md_path.stem, "words": len(plan.split())}
    files = sorted(p for p in run_dir.glob("P*.md"))
    with ThreadPoolExecutor(max_workers=len(files)) as pool:
        df = pd.DataFrame(list(pool.map(one, files)))
    mean = df.drop(columns=["project"]).mean(numeric_only=True).round(1).to_dict()
    board = EXPERIMENTS / "leaderboard.csv"
    pd.concat([pd.read_csv(board), pd.DataFrame([{"method": "prompting", "version": version, "run": run_dir.name,
                                                   "notes": notes, **mean}])]).to_csv(board, index=False)
    return df

# find the v1 run folder again after the restart, then rescore it
# run_v1 = sorted(EXPERIMENTS.glob("prompting_v1_*"))[-1]
# rescore(run_v1, "v1", notes="zero-shot baseline (rescored, evaluator fix)")

In [4]:
def rescore_all(tag=EVAL_TAG, workers=4):
    """Re-score every saved plan of every prompting run with the CURRENT evaluator.
    Run this ONLY after changing evaluate.py (and changing EVAL_TAG in Cell 1)."""
    jobs = [(d, f) for d in sorted(EXPERIMENTS.glob("prompting_*")) if d.is_dir() for f in sorted(d.glob("P*.md"))]
    print(len(jobs), "plans to score")
    def one(job):
        d, f = job
        plan = f.read_text(encoding="utf-8")
        r, cov = ev.score_plan(plan, f.stem[:3])
        cov.to_csv(d / f"{f.stem}_coverage.csv", index=False)
        (d / f"{f.stem}_scores.json").write_text(json.dumps(r, indent=2, default=str), encoding="utf-8")
        print("scored", d.name, f.stem[:3], flush=True)
        return {"version": d.name.split("_")[1], "run": d.name, "project": f.stem, "words": len(plan.split()),
                **ev.summary(r), "unverified_supported": r["unverified_supported"]}
    with ThreadPoolExecutor(max_workers=workers) as pool:
        rows = pd.DataFrame(list(pool.map(one, jobs)))
    rows["eval"] = tag
    path = EXPERIMENTS / "scores_by_plan.csv"
    rows.to_csv(path, mode="a", header=not path.exists(), index=False)
    return rows

# rows = rescore_all()     # <- run manually only after an evaluator change

In [5]:
def board(tag=EVAL_TAG):
    """Mean ± std per version over all runs and both projects, for one evaluator version."""
    s = pd.read_csv(EXPERIMENTS / "scores_by_plan.csv")
    s = s[s["eval"] == tag]
    cols = ["overall", "structure", "coverage", "faithfulness", "invented", "similarity", "words"]
    out = s.groupby("version")[cols].mean().round(1)
    out["overall_std"] = s.groupby("version")["overall"].std().round(1)
    out["plans"] = s.groupby("version").size()
    return out

board()

,overall,structure,coverage,faithfulness,invented,similarity,words,overall_std,plans
version,,,,,,,,,
v1,73.1,72.5,85.9,87.7,7.8,88.7,3439.2,5.5,8
v2,93.6,100.0,92.9,98.9,0.5,93.0,4175.8,1.0,4
v3,93.0,100.0,92.1,97.6,1.2,96.0,4081.0,2.0,4


In [6]:
V1_SYSTEM = "You are a senior QA lead. You write software test plans."

def v1_user(inp):
    return (f"Write a complete test plan for the project described below.\n\n"
            f"<brd>\n{inp['brd']}\n</brd>\n\n<architecture>\n{inp['arch']}\n</architecture>\n\n"
            "Output the test plan in Markdown with # headings.")

(PROMPTS / "v1_zero_shot.json").write_text(json.dumps({"system": V1_SYSTEM, "user": v1_user({"brd": "{brd}", "arch": "{arch}"})}, indent=2), encoding="utf-8")

df_v1, run_v1 = run_version("v1", V1_SYSTEM, v1_user, notes="zero-shot baseline")
df_v1

done P07_Email_Order_Entry: gen 47s, 3746 words
done P06_Rebate_Validation: gen 37s, 2883 words


,version,run,project,words,overall,structure,coverage,coverage_all,faithfulness,invented,similarity,embedding_sim,unverified_supported,eval
0,v1,prompting_v1_20261023_2309,P06_Rebate_Validation,2883,76.8,85.0,78.8,75.0,90.9,4,92.7,54.9,1,eval4
1,v1,prompting_v1_20261023_2309,P07_Email_Order_Entry,3746,77.1,80.0,86.6,84.8,89.6,7,90.5,50.7,3,eval4


In [7]:
print(leaderboard().to_string(index=False))
diagnose(run_v1, "P06")

version  overall  structure  coverage  faithfulness  invented  similarity  words                                        notes
     v1     54.0       35.0      85.6          88.9       5.5        46.4 3164.0                           zero-shot baseline
     v1     61.8       70.0      85.5          75.5      23.5        86.5 3164.0 zero-shot baseline (rescored, evaluator fix)
     v1     70.4       70.0      89.1          84.2       9.5        87.8 3696.0                           zero-shot baseline
     v2     89.2      100.0      90.6          95.2       5.5        92.9 4153.0  role + outline + grounding + coverage rules
     v3     79.6      100.0      90.6          99.2       0.5        48.1 4251.5                  v2 + one-shot example (P01)
     v1     69.2       65.0      88.6          83.4      10.5        89.8 3696.0 zero-shot baseline (rescored, evaluator fix)
     v1     68.8       80.0      85.9          79.8      19.5        90.6 3324.0                           zero-shot b

In [8]:
print((run_v1 / f"{ev.project_dir('P06').name}.md").read_text(encoding="utf-8")[:4000])

# Test Plan — Supplier Rebate Claim Validation

Version: 1.0  
Prepared by: QA Lead  
Date: 2026-02-XX

## 1. Document Purpose
This test plan describes the testing strategy, scope, resources, schedule, and detailed test approaches for Supplier Rebate Claim Validation (MVP1, with MVP2 test considerations). It ensures the product meets functional, non-functional, security, regulatory and acceptance criteria defined in the Business Requirement Specification.

## 2. Objectives
- Verify automated extraction of rebate agreement terms (tiers, rates, eligible product lines, periods) from PDF agreements using Azure Document Intelligence and Azure OpenAI meets ≥95% extraction accuracy compared to golden dataset.
- Ensure analyst confirmation flow prevents unconfirmed terms from being used.
- Validate correct rebate calculations (retroactive and incremental) against Databricks Delta Lake purchase data; meet 100% calculation accuracy on analyst-verified examples.
- Validate claim worksheet generat

In [9]:
TITLES = {
    "document_control": "Document Control", "references": "References", "objectives": "Test Objectives",
    "scope_in": "In Scope", "scope_out": "Out of Scope", "assumptions_dependencies": "Assumptions and Dependencies",
    "test_strategy": "Test Strategy", "functional_coverage": "Functional Test Coverage",
    "rbac_testing": "Roles, Permissions and Workflow Testing",
    "security_compliance": "Security, Privacy and Compliance Testing",
    "ai_data_quality": "AI and Data Quality Validation", "nfr_performance": "Performance Testing",
    "nfr_reliability": "Reliability and Recovery Testing",
    "nfr_accessibility_compat": "Accessibility and Compatibility Testing",
    "observability_audit": "Logging, Monitoring and Audit Testing",
    "test_data_environment": "Test Environments and Test Data",
    "entry_exit_criteria": "Entry, Suspension and Exit Criteria", "defect_management": "Defect Management",
    "risks": "Risks and Mitigations", "deliverables_reporting": "Test Deliverables and Reporting",
    "traceability_metrics": "Traceability and Metrics", "schedule": "Test Schedule",
    "e2e_scenario": "End-to-End Scenario", "approvals": "Approvals",
}
OUTLINE = "\n".join(f"# {i}. {TITLES[k]}\n   -> {ev.SECTIONS[k]}" for i, k in enumerate(ev.SECTIONS, 1))

V2_SYSTEM = """You are a senior QA lead writing the master test plan for the MVP release of an AI-enabled enterprise application. Business owners, architects and the QA team will review it.

GROUNDING RULES
1. Use only facts stated in the BRD and the architecture description. Copy numbers, units, thresholds, system names, role names and MVP labels exactly as written.
2. Never invent systems, vendors, test tools, environments, dates, volumes or targets. If the plan needs something the documents do not give (e.g. a test tool or test dates), write "TBD – confirm with <owner>".
3. Test approaches, techniques and example test data are your professional choice, but they must be consistent with the documents.
4. MVP1 is in scope. MVP2 items and explicit exclusions go to Out of Scope, listed item by item.

COVERAGE RULES – address every one of these that appears in the documents:
- every MVP1 functional requirement and user story: what will be verified
- every business rule and threshold (amounts, percentages, approval limits, tolerances)
- every NFR target (availability, performance, volumes, timings) with its exact value
- every role and its permissions, including segregation of duties
- every integration / connection in the architecture (source -> target, protocol / auth)
- every data source with its owner, classification and refresh frequency
- AI / extraction quality: confidence thresholds, human review, controls against hallucination
- assumptions, dependencies, risks, acceptance criteria and timeline

FORMAT
- Markdown. Use exactly the H1 headings of the outline, in that order, with those names. Use H2 for sub-features.
- No table of contents. Prefer bullets and tables. No filler."""

def v2_user(inp):
    return (f"<brd>\n{inp['brd']}\n</brd>\n\n<architecture>\n{inp['arch']}\n</architecture>\n\n"
            f"OUTLINE (the text after -> explains what each section must contain; do not copy it):\n{OUTLINE}\n\n"
            "Write the complete test plan now.")

(PROMPTS / "v2_structured_grounded.json").write_text(
    json.dumps({"system": V2_SYSTEM, "user": v2_user({"brd": "{brd}", "arch": "{arch}"})}, indent=2), encoding="utf-8")

# df_v2, run_v2 = run_version("v2", V2_SYSTEM, v2_user, notes="role + outline + grounding + coverage rules")   # already done
run_v2 = sorted(EXPERIMENTS.glob("prompting_v2_*"))[-1]
print("v2 run:", run_v2.name)

v2 run: prompting_v2_20261023_1929


In [10]:
print(leaderboard().to_string(index=False))
diagnose(run_v2, "P06")

version  overall  structure  coverage  faithfulness  invented  similarity  words                                        notes
     v1     54.0       35.0      85.6          88.9       5.5        46.4 3164.0                           zero-shot baseline
     v1     61.8       70.0      85.5          75.5      23.5        86.5 3164.0 zero-shot baseline (rescored, evaluator fix)
     v1     70.4       70.0      89.1          84.2       9.5        87.8 3696.0                           zero-shot baseline
     v2     89.2      100.0      90.6          95.2       5.5        92.9 4153.0  role + outline + grounding + coverage rules
     v3     79.6      100.0      90.6          99.2       0.5        48.1 4251.5                  v2 + one-shot example (P01)
     v1     69.2       65.0      88.6          83.4      10.5        89.8 3696.0 zero-shot baseline (rescored, evaluator fix)
     v1     68.8       80.0      85.9          79.8      19.5        90.6 3324.0                           zero-shot b

In [11]:
EXAMPLE = "P01"                                   # TRAINING project only – never validation/test
assert ev.project_dir(EXAMPLE).parent.name == "train"

def render_gold_plan(prefix):
    """Reference plan of a training project, re-ordered into our 24-section outline."""
    gold = json.loads((ev.project_dir(prefix, ev.GOLD) / "gold_testplan.json").read_text(encoding="utf-8"))
    parts = []
    for i, k in enumerate(ev.SECTIONS, 1):
        sec = gold["sections"][k]
        parts.append(f"# {i}. {TITLES[k]}\n" + (sec["text"] if sec["present"] else "(not covered in this example)"))
    return "\n\n".join(parts)

EX = inputs(EXAMPLE)
EX_PLAN = render_gold_plan(EXAMPLE)
print(EX["name"], "| example plan words:", len(EX_PLAN.split()))

V3_SYSTEM = V2_SYSTEM + """

WORKED EXAMPLE
You also receive one example: the BRD and architecture of a DIFFERENT project and the test plan a senior QA lead wrote for it.
Match its depth and the kind of test content it contains: concrete test conditions, worked numeric examples derived from the BRD's own rules, edge cases, explicit security / accessibility / audit checks, and measurable entry and exit criteria.
NEVER copy facts, names, numbers or systems from the example. Every project-specific fact in your plan must come from the NEW project's documents."""

def v3_user(inp):
    return (f"<example>\n<example_brd>\n{EX['brd']}\n</example_brd>\n"
            f"<example_architecture>\n{EX['arch']}\n</example_architecture>\n"
            f"<example_test_plan>\n{EX_PLAN}\n</example_test_plan>\n</example>\n\n"
            f"NEW PROJECT\n<brd>\n{inp['brd']}\n</brd>\n\n<architecture>\n{inp['arch']}\n</architecture>\n\n"
            f"OUTLINE (the text after -> explains what each section must contain; do not copy it):\n{OUTLINE}\n\n"
            "Write the complete test plan for the NEW project now.")

(PROMPTS / "v3_few_shot.json").write_text(
    json.dumps({"system": V3_SYSTEM, "example": EXAMPLE, "user": v3_user({"brd": "{brd}", "arch": "{arch}"})[-1500:]},
               indent=2), encoding="utf-8")

# df_v3, run_v3 = run_version("v3", V3_SYSTEM, v3_user, notes=f"v2 + one-shot example ({EXAMPLE})")   # already done
run_v3 = sorted(EXPERIMENTS.glob("prompting_v3_*"))[-1]
print("v3 run:", run_v3.name)

P01_Invoice_Matching | example plan words: 1738
v3 run: prompting_v3_20261023_1942


In [12]:
GENERIC = {"BRD", "NFR", "NFRs", "SLA", "SLAs", "UAT", "MFA", "API", "APIs", "PDF", "PDFs", "IDs", "MVP", "MVP1",
           "MVP2", "RBAC", "SSO", "UI", "QA", "AI", "Audit", "Security", "Data", "Owner", "Azure"}

def names(text):
    """Proper names only: capitalised words mid-sentence, multi-word Capitalised names, ACRONYMS."""
    mid = re.findall(r"(?<=[a-z0-9,)] )([A-Z][A-Za-z0-9&+-]{2,}(?: [A-Z][A-Za-z0-9&+-]{2,})*)", text)
    acr = re.findall(r"\b[A-Z][A-Z0-9]{1,}[a-z]?\b", text)
    return (set(mid) | set(acr)) - GENERIC

def leak_check(run_dir, prefix, example=EXAMPLE):
    target = inputs(prefix)
    src = (target["brd"] + " " + target["arch"]).lower()
    only_in_example = {t for t in names(inputs(example)["brd"]) if t.lower() not in src}
    plan = (run_dir / f"{target['name']}.md").read_text(encoding="utf-8")
    hits = sorted(t for t in only_in_example if re.search(r"\b" + re.escape(t) + r"\b", plan))
    print(f"{target['name']}: {len(hits)} example-only names in plan -> {hits}")

for p in VAL:
    leak_check(run_v3, p)
diagnose(run_v3, "P06")

P06_Rebate_Validation: 1 example-only names in plan -> ['Ops']


P07_Email_Order_Entry: 1 example-only names in plan -> ['Ops']
Missing sections: []

Coverage by category: {'acceptance': 1.0, 'ai_quality': 1.0, 'architecture': 0.88, 'assumption': 1.0, 'business_rule': 0.57, 'compliance': 1.0, 'data': 0.6, 'dependency': 1.0, 'functional': 1.0, 'integration': 0.87, 'nfr_availability': 1.0, 'nfr_performance': 0.67, 'out_of_scope': 1.0, 'risk': 1.0, 'role': 1.0, 'scope_mvp1': 0.83, 'scope_mvp2': 1.0, 'security_access': 0.83, 'timeline': 0.5}

Invented:
 - {'claim': 'Audit trail must record who confirmed terms and timestamp', 'kind': 'requirement', 'verdict': 'supported'}

Missed must-test facts:
 - [scope_mvp1] MVP1 calculates earned rebates from sales and purchase data in the enterprise data lake
 - [nfr_performance] Cut claim preparation time from 3 days to 4 hours per supplier
 - [data] Calculations data source is Tool output and frequency is Per run
 - [data] Calculations owner is Supplier Programs and security classification is Confidential
 - [int

In [13]:
AREAS = ["functional", "business_rule", "worked_example", "nfr", "kpi", "role_permission", "integration", "data",
         "ai_quality", "out_of_scope", "assumption", "dependency", "risk", "acceptance", "timeline"]

BASIS_SYSTEM = (
    "You are a senior QA analyst preparing the TEST BASIS for a test plan. From the BRD and the architecture "
    "description, list every item a test plan must address, as short atomic statements that keep exact values and names. "
    "Include: each MVP1 functional requirement and user story; each business rule, threshold and approval limit; "
    "each worked numeric example in the documents (as a test vector: inputs and expected result); each NFR target AND "
    "each business KPI / success metric with its exact value; each role and its permissions; each architecture "
    "connection (source -> target, protocol / auth); each data source with owner, classification and frequency; "
    "AI confidence and human-review rules; each MVP2 or excluded item; assumptions, dependencies, risks, acceptance "
    "criteria and milestones. Do not invent anything.")
BASIS_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["items"],
    "properties": {"items": {"type": "array", "items": {
        "type": "object", "additionalProperties": False, "required": ["area", "mvp", "statement"],
        "properties": {"area": {"type": "string", "enum": AREAS},
                       "mvp": {"type": "string", "enum": ["MVP1", "MVP2", "All", "NA"]},
                       "statement": {"type": "string"}}}}}}

BASIS_LOG = {}                                        # project -> (basis items, IDs missing from the plan)

def test_basis(inp):
    out = ask_json(BASIS_SYSTEM, f"<brd>\n{inp['brd']}\n</brd>\n\n<architecture>\n{inp['arch']}\n</architecture>",
                   BASIS_SCHEMA, "test_basis", effort="medium")
    items = out["items"]
    for i, it in enumerate(items, 1):
        it["id"] = f"T{i:02d}"
    return items

V4_SYSTEM = V3_SYSTEM + """

TEST BASIS
You also receive a numbered TEST BASIS (T01, T02, ...) extracted from the NEW project's documents. Address EVERY item:
worked examples become concrete test cases with the expected result; KPIs and NFR targets become measurable checks;
MVP2 and excluded items go to Out of Scope.
In the section "Traceability and Metrics" include a table | Test basis ID | Item (short) | Plan section | listing every ID."""

def v4_generate(inp):
    basis = test_basis(inp)                                            # call 1: what must be covered
    listing = "\n".join(f"{b['id']} [{b['area']}|{b['mvp']}] {b['statement']}" for b in basis)
    user = v3_user(inp).replace("Write the complete test plan for the NEW project now.",
                                f"TEST BASIS:\n{listing}\n\nWrite the complete test plan for the NEW project now.")
    plan = ask(V4_SYSTEM, user, effort="medium")                       # call 2: write the plan
    missing = [b["id"] for b in basis if not re.search(rf"\b{b['id']}\b", plan)]
    BASIS_LOG[inp["name"]] = (basis, missing)
    return plan

(PROMPTS / "v4_test_basis.json").write_text(
    json.dumps({"basis_system": BASIS_SYSTEM, "system": V4_SYSTEM}, indent=2), encoding="utf-8")

df_v4, run_v4 = run_version("v4", V4_SYSTEM, generate=v4_generate, notes="v3 + test basis first + traceability")
for name, (basis, missing) in BASIS_LOG.items():
    print(f"{name}: {len(basis)} basis items, {len(missing)} not referenced in plan -> {missing}")
df_v4

done P07_Email_Order_Entry: gen 97s, 5344 words
done P06_Rebate_Validation: gen 101s, 4945 words
P07_Email_Order_Entry: 91 basis items, 0 not referenced in plan -> []
P06_Rebate_Validation: 68 basis items, 0 not referenced in plan -> []


,version,run,project,words,overall,structure,coverage,coverage_all,faithfulness,invented,similarity,embedding_sim,unverified_supported,eval
0,v4,prompting_v4_20261023_2314,P06_Rebate_Validation,4945,92.3,100.0,93.1,91.8,96.4,2,96.4,71.7,2,eval4
1,v4,prompting_v4_20261023_2314,P07_Email_Order_Entry,5344,92.3,100.0,96.4,95.7,95.8,3,93.3,71.2,3,eval4


In [14]:
print(board().to_string())
diagnose(run_v4, "P06")

         overall  structure  coverage  faithfulness  invented  similarity   words  overall_std  plans
version                                                                                              
v1          73.9       74.5      85.3          88.2       7.3        89.3  3414.3          5.2     10
v2          93.6      100.0      92.9          98.9       0.5        93.0  4175.8          1.0      4
v3          93.0      100.0      92.1          97.6       1.2        96.0  4081.0          2.0      4
v4          92.3      100.0      94.8          96.1       2.5        94.8  5144.5          0.0      2
Missing sections: []

Coverage by category: {'acceptance': 1.0, 'ai_quality': 1.0, 'architecture': 0.75, 'assumption': 1.0, 'business_rule': 0.71, 'compliance': 1.0, 'data': 0.65, 'dependency': 1.0, 'functional': 1.0, 'integration': 0.96, 'nfr_availability': 1.0, 'nfr_performance': 1.0, 'out_of_scope': 1.0, 'risk': 1.0, 'role': 0.92, 'scope_mvp1': 1.0, 'scope_mvp2': 1.0, 'security_acce